# Quick, Draw! Animal Classifier

## Setup and data

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import matplotlib.pyplot as plt
from src.data import animals, load_data, train_test_split
from src.pca import PCA

X, y = load_data(animals)
print(X.shape, y.shape)
print(np.bincount(y))
print(X.min(), X.max())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y)
print(X_train.shape, X_test.shape)
print(np.bincount(y_train))

## Phase 1: Explore the drawings

In [ ]:
i = 12345
plt.imshow(X[i].reshape(28, 28), cmap="gray_r")
plt.title(animals[y[i]])
plt.show()

In [ ]:
fig, axes = plt.subplots(len(animals), 8, figsize=(10, 13))
for c, name in enumerate(animals):
    examples = X_train[y_train == c][:8]
    for j in range(8):
        axes[c, j].imshow(examples[j].reshape(28, 28), cmap="gray_r")
        axes[c, j].axis("off")
    axes[c, 0].set_title(name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for c, ax in enumerate(axes.flat):
    mean_img = X_train[y_train == c].mean(axis=0)
    ax.imshow(mean_img.reshape(28, 28), cmap="gray_r")
    ax.set_title(animals[c])
    ax.axis("off")
plt.show()

## Phase 2: PCA

In [ ]:
pca = PCA(k=150).fit(X_train)

cum = np.cumsum(pca.explained)
plt.plot(np.arange(1, len(cum) + 1), cum)
plt.xlabel("Number of components")
plt.ylabel("Cumulative explained variance")
plt.grid(True)
plt.show()

for target in [0.8, 0.9, 0.95]:
    print(f"{target:.0%} of variance: {np.argmax(cum >= target) + 1} components")

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=(12, 4.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(pca.components[i].reshape(28, 28), cmap="RdBu")
    ax.set_title(f"PC {i+1}")
    ax.axis("off")
plt.show()

In [ ]:
Z = pca.transform(X_train)
plt.figure(figsize=(9, 7))
for c, name in enumerate(animals):
    pts = Z[y_train == c][:300]
    plt.scatter(pts[:, 0], pts[:, 1], s=4, alpha=0.5, label=name)
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.legend(markerscale=4)
plt.show()